In [1]:
from typing import List, Optional
import numpy as np 
import pandas as pd 
import os
from abc import ABC, abstractmethod

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split

# Pre-process Data Methods

In [ ]:
german_path = os.path.join('..', 'datasets', 'german', 'german.data')
columns = [
    'CheckingAccount',
    'Duration',
    'CreditHistory',
    'Purpose',
    'CreditAmount',
    'SavingsAccount',
    'EmploymentSince',
    'InstallmentRate',
    'PersonalStatusAndSex',
    'OtherDebtors',
    'ResidenceSince',
    'Property',
    'Age',
    'OtherInstallmentPlans',
    'Housing',
    'ExistingCredits',
    'Job',
    'Dependents',
    'Telephone',
    'ForeignWorker',
    'Target'
]

try:
    print(f'Importing Data from German Dataset...')
    df = pd.read_csv(
        german_path,
        header=None,
        sep=r'\s+',
        engine='python',
        names=columns
    )
    y_raw = df['Target']
    y = y_raw - 1
    X = df.drop(columns=['Target'])

    numerical_cols = X.select_dtypes(include=['number']).columns
    categorical_cols = X.select_dtypes(exclude=['number']).columns

    # Perform One-Hot Encode
    X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
    bool_cols = [col for col in X.columns if X[col].dtype == 'bool']
    for col in bool_cols:
        X[col] = X[col].astype(float)

    # Scale Numerical Values
    scaler = StandardScaler()
    X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

    X['Target'] = y

    german_preprocessed_path = os.path.join('..', 'data', 'preprocessed', 'german.csv')
    output_dir = os.path.dirname(german_preprocessed_path)
    os.makedirs(output_dir, exist_ok=True)

    X.to_csv(german_preprocessed_path, index=False)
except FileNotFoundError:
    raise ValueError(f'Path not found for German Dataset: {german_path}')

Importing Data from German Dataset...


# Dataloader/Data

In [3]:
class Dataset(ABC):
    def __init__(
        self, 
        name : str, 
        path : str,
        test_size : float = 0.2,
        random_state : Optional[int] = None
    ) -> None:
        self.name = name
        self.path = path
        self.random_state = random_state
        self.test_size = test_size

        self.data = None
        self.X_train = None
        self.y_train = None 
        self.X_test = None 
        self.y_test = None

        self.load_data()
        self.train_test_split(test_size=test_size, random_state=random_state)

    @abstractmethod
    def load_data(self) -> None:
        pass

    def train_test_split(self, test_size: float, random_state: Optional[int]) -> None:
        df: pd.DataFrame = self.data
        self.X = df.drop(columns=['Target'])
        self.y = df['Target']

        (
            self.X_train,
            self.X_test,
            self.y_train,
            self.y_test
        ) = train_test_split(
            self.X,
            self.y,
            test_size=self.test_size,
            random_state=self.random_state
        )

class German(Dataset):
    def __init__(
        self,
        path : str,
        random_state : Optional[int] = None
    ):
        super().__init__("German", path, random_state)
    
    def load_data(self) -> None:
        self.data = pd.read_csv(self.path)

# Testing Environment:

In [2]:
from Temis.preprocess_data import preprocess_german
from Temis.dataloader import German

german_path = os.path.join('..', 'datasets', 'german', 'german.data')
preprocess_german(german_path)
# german_preprocessed_path (gpp_path)
gpp_path = os.path.join('..', 'data', 'preprocessed', 'german.csv')
german = German(gpp_path)


Starting preprocess_german
Importing Data from German Dataset...
